In [ ]:
# Install transformers from the PR branch until it's merged! using python 3.12.7
#https://huggingface.co/lightonai/LightOnOCR-1B-1025
!pip install -q -U git+https://github.com/baptiste-aubertin/transformers.git@main
!pip install torch transformers torchvision accelerate

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, LightOnOCRForConditionalGeneration
import requests
from io import BytesIO

# Load Model
model_id = "lightonai/LightOnOCR-1B-1025"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
model = LightOnOCRForConditionalGeneration.from_pretrained(model_id, dtype=torch.bfloat16, device_map=device, attn_implementation="sdpa")
model.eval();

In [ ]:
# Get a test image
image_url = "https://jeroen.github.io/images/testocr.png"
response = requests.get(image_url)
image = Image.open(BytesIO(response.content)).convert("RGB")

display(image)

# Run inference
messages = [{"role": "user", "content": [{"type": "image"}]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = processor(text=[text], images=[image], return_tensors="pt").to(device)
inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
)

input_length = inputs['input_ids'].shape[1]
generated_text = processor.tokenizer.decode(outputs[0, input_length:], skip_special_tokens=True)

print("============\n")
print(generated_text)